### 2. ✅ Role-Based Access Control — (Django/FastAPI context)
Maps directly to real backend work.

**Why it impresses**: directly maps to JWT/auth pipelines they'll ask about, shows you think in terms of security and architecture.

In [ ]:
import functools
from typing import Callable

# Simulated auth context (in Django/FastAPI this comes from request/token)
class PermissionError(Exception):
    pass

def require_roles(*allowed_roles: str):
    """
    Decorator factory for role-based access control.
    Real use case: Django views, FastAPI endpoints, service layer methods.
    """
    def decorator(func: Callable):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            # In real Django: request.user.role
            # In FastAPI: injected via Depends(get_current_user)
            user = kwargs.get("user") or (args[0] if args else None)

            if not user:
                raise PermissionError("Authentication required.")

            if user.role not in allowed_roles:
                raise PermissionError(
                    f"Role '{user.role}' not authorized. "
                    f"Required: {allowed_roles}"
                )
            return func(*args, **kwargs)
        return wrapper
    return decorator

In [ ]:
# --- Usage ---
from dataclasses import dataclass

@dataclass
class User:
    name: str
    role: str

@require_roles("admin", "moderator")
def delete_post(user: User, post_id: int):
    print(f"{user.name} deleted post {post_id}")

@require_roles("admin")
def ban_user(user: User, target_id: int):
    print(f"{user.name} banned user {target_id}")

In [ ]:
# Test it
alice = User("Alice", "admin")
bob   = User("Bob",   "viewer")

delete_post(user=alice, post_id=42)   # ✅ works
delete_post(user=bob,   post_id=42)   # ❌ raises PermissionError

### Full Django Example — RBAC Decorator

In [ ]:
# permissions.py
import functools
from django.http import JsonResponse
from django.contrib.auth.decorators import login_required


def require_roles(*allowed_roles: str):
    """
    Decorator factory for Django views.
    Works on function-based views AND as a base for class-based views.

    Usage:
        @require_roles("admin", "moderator")
        def delete_post(request, post_id): ...
    """
    def decorator(view_func):
        @functools.wraps(view_func)                    # preserves view metadata
        @login_required                                # inner-most runs first
        def wrapper(request, *args, **kwargs):
            # Django stores roles/groups in request.user
            user_groups = set(
                request.user.groups.values_list("name", flat=True)
            )
            if not user_groups.intersection(allowed_roles):
                return JsonResponse(
                    {
                        "error": "Forbidden",
                        "detail": f"Required role(s): {list(allowed_roles)}"
                    },
                    status=403
                )
            # ← passes the check → call the actual view
            return view_func(request, *args, **kwargs)
        return wrapper
    return decorator


# ── Method decorator for Class-Based Views ─────────────────────
from django.utils.decorators import method_decorator

def rbac(*allowed_roles: str):
    """Thin wrapper so the same logic works on CBV methods."""
    return method_decorator(require_roles(*allowed_roles))

In [ ]:
# views.py
from django.http import JsonResponse
from django.views import View
from .permissions import require_roles, rbac


# ── Function-Based Views ───────────────────────────────────────

@require_roles("admin")
def delete_user(request, user_id: int):
    # Only reached if user is authenticated AND has "admin" group
    return JsonResponse({"deleted": user_id})


@require_roles("admin", "moderator")
def moderate_post(request, post_id: int):
    return JsonResponse({"moderated": post_id})


@require_roles("viewer")                    # ← least privileged
def list_posts(request):
    return JsonResponse({"posts": []})


# ── Class-Based Views ──────────────────────────────────────────

@method_decorator(require_roles("admin"), name="dispatch")
class AdminDashboardView(View):
    """Entire view class protected — ALL HTTP methods require admin."""
    def get(self, request):
        return JsonResponse({"dashboard": "admin data"})


class PostView(View):
    """Per-method protection — GET is open, DELETE requires admin."""

    def get(self, request, post_id: int):
        return JsonResponse({"post": post_id})

    @rbac("admin")
    def delete(self, request, post_id: int):
        return JsonResponse({"deleted": post_id})

In [ ]:
# urls.py
from django.urls import path
from . import views

urlpatterns = [
    path("users/<int:user_id>/delete/", views.delete_user),
    path("posts/<int:post_id>/moderate/", views.moderate_post),
    path("posts/", views.list_posts),
    path("admin/dashboard/", views.AdminDashboardView.as_view()),
    path("posts/<int:post_id>/", views.PostView.as_view()),
]

**Django Execution Flow**
```shell
Request hits delete_user
        │
        ▼
wrapper(request, user_id)            ← functools.wraps wrapper
        │
        ▼
@login_required check                ← is request.user authenticated?
   NO  → redirect to /login/
   YES ↓
        │
        ▼
user_groups = {"editor", "viewer"}   ← fetch from DB (cached by Django)
        │
        ▼
{"editor","viewer"} ∩ {"admin"} = ∅  ← intersection check
        │
   empty → return 403 JsonResponse
   match → call delete_user(request, user_id)  ← original view runs
```

---

### Full FastAPI Example — Your Pattern + Decorator Pattern Side by Side

In [ ]:
# models.py
from enum import Enum
from pydantic import BaseModel


class UserGroupEnum(str, Enum):
    ADMIN     = "admin"
    MODERATOR = "moderator"
    VIEWER    = "viewer"


class User(BaseModel):
    id:       int
    username: str
    groups:   list[UserGroupEnum]

    def has_group(self, role: UserGroupEnum) -> bool:
        return role in self.groups

In [ ]:
# auth.py
from fastapi import HTTPException, status
from fastapi.security import OAuth2PasswordBearer
from .models import User, UserGroupEnum

oauth2_scheme = OAuth2PasswordBearer(tokenUrl="/auth/token")

# Simulated DB — in production this decodes JWT + hits DB
FAKE_USERS = {
    "alice_token": User(id=1, username="Alice", groups=[UserGroupEnum.ADMIN]),
    "bob_token":   User(id=2, username="Bob",   groups=[UserGroupEnum.MODERATOR]),
    "eve_token":   User(id=3, username="Eve",   groups=[UserGroupEnum.VIEWER]),
}

async def get_current_user(token: str = Depends(oauth2_scheme)) -> User:
    user = FAKE_USERS.get(token)
    if not user:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Invalid or expired token",
            headers={"WWW-Authenticate": "Bearer"},
        )
    return user

In [ ]:
# dependencies.py  ← YOUR PATTERN (closure / dependency factory)
from typing import Annotated
from fastapi import Depends, HTTPException, status
from .models import User, UserGroupEnum
from .auth import get_current_user


def require_role(*allowed_roles: UserGroupEnum):
    """
    Dependency factory — closure over allowed_roles.
    FastAPI calls role_checker(current_user=<injected>) on every request.

    The dependency graph FastAPI builds:
      route handler
        └── Depends(require_role(ADMIN))
              └── role_checker(current_user)
                    └── Depends(get_current_user)
                          └── OAuth2 token from header
    """
    def role_checker(
        current_user: Annotated[User, Depends(get_current_user)]
    ) -> User:
        # 'allowed_roles' closed over from outer scope
        if not any(current_user.has_group(role) for role in allowed_roles):
            allowed = ", ".join(r.value for r in allowed_roles)
            raise HTTPException(
                status_code=status.HTTP_403_FORBIDDEN,
                detail=f"Access denied. Required role(s): {allowed}",
            )
        return current_user   # returned user can be re-used in route handler

    return role_checker

In [ ]:
# routers/posts.py
from fastapi import APIRouter, Depends
from typing import Annotated
from ..models import User, UserGroupEnum
from ..dependencies import require_role

router = APIRouter(prefix="/posts", tags=["posts"])

# Reusable dependency aliases — cleaner route signatures
AdminUser     = Annotated[User, Depends(require_role(UserGroupEnum.ADMIN))]
ModeratorUser = Annotated[User, Depends(
    require_role(UserGroupEnum.ADMIN, UserGroupEnum.MODERATOR)
)]


# ── Pattern 1: dependencies=[...] — user NOT available in handler ──
@router.delete(
    "/{post_id}",
    dependencies=[Depends(require_role(UserGroupEnum.ADMIN))]
)
async def delete_post(post_id: int):
    # We only need to GATE access — don't need the user object itself
    return {"deleted": post_id}


# ── Pattern 2: typed parameter — user IS available in handler ──────
@router.put("/{post_id}")
async def edit_post(post_id: int, current_user: ModeratorUser):
    # current_user is the verified User object returned by role_checker
    return {
        "edited_by": current_user.username,
        "post_id":   post_id
    }


# ── Pattern 3: inline Depends — most explicit ──────────────────────
@router.post("/")
async def create_post(
    current_user: Annotated[User, Depends(require_role(UserGroupEnum.ADMIN))]
):
    return {"created_by": current_user.username}

In [ ]:
# main.py
from fastapi import FastAPI
from .routers import posts

app = FastAPI()
app.include_router(posts.router)

### FastAPI Dependency Injection — What Actually Happens Per Request
```shell
DELETE /posts/42
Headers: Authorization: Bearer alice_token
        │
        ▼
FastAPI sees: dependencies=[Depends(require_role(ADMIN))]
        │
        ▼
require_role(ADMIN) was already called at startup → returned role_checker closure
        │
        ▼
FastAPI calls: role_checker(current_user=???)
        │
        ▼
FastAPI sees role_checker needs: Depends(get_current_user)
        │
        ▼
FastAPI calls: get_current_user(token=???)
        │
        ▼
FastAPI sees OAuth2PasswordBearer → extracts "alice_token" from header
        │
        ▼
get_current_user("alice_token") → returns User(Alice, groups=[ADMIN])
        │
        ▼
role_checker(current_user=Alice) →
    any(Alice.has_group(ADMIN)) → True ✅
        │
        ▼
delete_post(post_id=42) runs → {"deleted": 42}


── If Bob (MODERATOR) hits DELETE /posts/42 ──────────────────

role_checker(current_user=Bob) →
    any(Bob.has_group(ADMIN)) → False ❌
        │
        ▼
raises HTTPException(403, "Access denied. Required role(s): admin")
        │
FastAPI catches it → returns HTTP 403 response
delete_post() is NEVER called
```

---

**The Key Comparison — Interview-Ready Answer**
```shell
YOUR FastAPI pattern          Django decorator pattern
──────────────────────────    ────────────────────────────────
Closure ✓                     Closure ✓ (all decorators are)
NOT a decorator ✓             IS a decorator ✓ (wraps a callable)
FastAPI resolves deps         You manually call the view
No wrapper function           Has wrapper function
Returns a dependency          Returns a wrapped view function
Injected by the framework     Applied at definition time with @
Composable in dep graph       Stacked with multiple @decorators
Declared at route level       Declared at function level
```
The interview answer:

> "In FastAPI I use a dependency factory — a closure that captures configuration but isn't a decorator because it doesn't wrap a callable. FastAPI's DI system handles the injection. In Django, the same concept becomes a true decorator because I need to wrap the view function and intercept the request manually."